# 🛡️ AI Trust 安全掃描

**填寫下方表單 → 點 ▶ 按鈕 → 5–30 分鐘後自動下載 PDF 報告**

🔒 您的 API Key 從這個瀏覽器直接送到您的 AI 端點，AI Trust 完全看不到。

---

### 🎫 還沒有 License Token？

License Token 是 AI Trust 寄給您的一串 `eyJhbGci...` 字串，用來授權您執行探針。

**申請方式（任選其一）**：

| 方案 | 內容 | 費用 | 怎麼申請 |
|---|---|---|---|
| 🆓 **試用版** | 24h / 1 次 / 提示注入探針 | 免費 | 寄信至 [aitrust.tw@gmail.com](mailto:aitrust.tw@gmail.com)，主旨「申請試用」 |
| 🥉 **稽核版** | 7 天 / 3 次 / 9 大探針 | NT$ 8,000 | 至 [ai-trust.vercel.app/pricing](https://ai-trust.vercel.app/pricing) 選方案 |
| 🥈 **月卡版**（最多人選）| 30 天 / 20 次 / 9 大探針 | NT$ 25,000 | 同上 |
| 🥇 **年卡版** | 365 天 / 200 次 / 9 大探針 | NT$ 200,000 | 同上 |

🎉 **早鳥優惠**：前 10 位客戶享 8 折

收到信後 1 個工作日內，我們會寄您 License Token 與使用說明。


In [ ]:
# @title ▶ AI Trust 掃描（填好下方欄位後點此按鈕）
# @markdown ### 📋 第 1 步：貼上您的 License Token
LICENSE_TOKEN = "" # @param {type:"string"}
# @markdown ### 🤖 第 2 步：您要測試的 AI 端點 URL
ENDPOINT_URL = "https://api.openai.com/v1/chat/completions" # @param {type:"string"}
# @markdown ### 🔑 第 3 步：該端點的 API Key（不會離開這個瀏覽器）
API_KEY = "" # @param {type:"string"}
# @markdown ### 🧪 第 4 步：模型識別碼
MODEL = "gpt-4o-mini" # @param {type:"string"}
# @markdown ### ⚙️ 第 5 步：端點類型
PROVIDER = "openai_compatible" # @param ["openai_compatible", "anthropic", "custom"]
# @markdown ---
# @markdown ### 📊 第 6 步：要跑哪些探針？（依您的方案勾選）
prompt_injection = True       # @param {type:"boolean"}
bias = True                   # @param {type:"boolean"}
adversarial = True            # @param {type:"boolean"}
hallucination = True          # @param {type:"boolean"}
infringement = True           # @param {type:"boolean"}
agent_security = True         # @param {type:"boolean"}
rag_security = True           # @param {type:"boolean"}
mcp_security = True           # @param {type:"boolean"}
multimodal_security = True    # @param {type:"boolean"}
# @markdown ---
# @markdown 💡 **填好以上欄位後**，點本區塊最上方的 ▶ 按鈕開始掃描。第一次跑會自動安裝工具（約 30 秒）。

import base64, os, sys, subprocess, time, json, logging
from pathlib import Path

# 把 Colab 內建的 root logger 關掉，避免 ERROR:root 那種開發者訊息出現
logging.getLogger().setLevel(logging.CRITICAL)

SUPPORT_EMAIL = "aitrust.tw@gmail.com"

def _say_error(title, fix_hint=""):
    """顯示給客戶看的友善錯誤訊息，不含任何 Python traceback."""
    print()
    print("━" * 60)
    print(f"  ❌ {title}")
    if fix_hint:
        print(f"  → {fix_hint}")
    print("━" * 60)
    print(f"  仍有問題？請寄信至 {SUPPORT_EMAIL}")
    print(f"  附上參考代碼：E{int(time.time())}")

def _main():
    # ── 0. 檢查輸入是否完整 ──────────────────────────────────────────
    problems = []
    if not LICENSE_TOKEN or not LICENSE_TOKEN.startswith("eyJ"):
        problems.append("License Token 看起來不對（應該以 eyJ 開頭）— 請檢查是否完整貼上、沒有缺字或多空白")
    if not ENDPOINT_URL.startswith("http"):
        problems.append("AI 端點 URL 必須以 http:// 或 https:// 開頭")
    if not API_KEY:
        problems.append("API Key 不能空白")
    if problems:
        print()
        print("━" * 60)
        print("  ⚠️  請修正以下問題後重新點 ▶")
        print("━" * 60)
        for p in problems:
            print(f"   • {p}")
        return

    # ── 1. 首次使用：自動安裝掃描工具 ───────────────────────────────
    try:
        import ai_trust_scanner  # noqa: F401
    except ImportError:
        print("首次使用，正在準備掃描工具（約 30 秒）⋯")
        _WHEEL_B64 = "..."
        # 注意：wheel filename 必須符合 PEP 427 規範，不能隨便改
        _whl = Path("/tmp/ai_trust_scanner-1.0.0-py3-none-any.whl")
        try:
            _whl.write_bytes(base64.b64decode(_WHEEL_B64))
            _r = subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                 str(_whl), "reportlab>=4.0"],
                capture_output=True, text=True, timeout=120,
            )
            if _r.returncode != 0:
                _say_error("掃描工具準備失敗",
                           "請按 Colab 上方選單「代碼執行程序 → 重新啟動」後再試一次")
                return
        except Exception:
            _say_error("掃描工具準備失敗",
                       "請按 Colab 上方選單「代碼執行程序 → 重新啟動」後再試一次")
            return
        print("✅ 準備完成\n")

    try:
        from ai_trust_scanner.license import validate_license, print_license_banner
        from ai_trust_scanner.http_client import EndpointClient
        from ai_trust_scanner.runner import run_scan
        from ai_trust_scanner.report import save_json, save_pdf
    except ImportError:
        _say_error("掃描工具載入失敗",
                   "請按 Colab 上方選單「代碼執行程序 → 重新啟動」後再試一次")
        return

    # ── 2. 驗證 License ──────────────────────────────────────────────
    try:
        lic = validate_license(LICENSE_TOKEN)
    except SystemExit:
        # validate_license 內部呼叫 _die() 會 raise SystemExit，attached errno is the message
        _say_error("License Token 無效",
                   "請確認您完整貼上了 Token，或寄信申請新的 Token")
        return
    except Exception as e:
        msg = str(e).lower()
        if "expired" in msg:
            _say_error("您的 License 已過期", "請寄信申請續約")
        else:
            _say_error("License 驗證失敗", "請確認 Token 是否正確完整")
        return
    print_license_banner(lic)

    # ── 3. 篩選探針 ──────────────────────────────────────────────────
    selected = {
        "prompt_injection": prompt_injection,
        "bias": bias,
        "adversarial": adversarial,
        "hallucination": hallucination,
        "infringement": infringement,
        "agent_security": agent_security,
        "rag_security": rag_security,
        "mcp_security": mcp_security,
        "multimodal_security": multimodal_security,
    }
    chosen = [name for name, on in selected.items() if on]
    allowed = set(lic.packs)
    will_run = [p for p in chosen if p in allowed]
    will_skip = [p for p in chosen if p not in allowed]

    if will_skip:
        print(f"ℹ️  以下探針不在您的方案內，已自動跳過：{', '.join(will_skip)}")
    if not will_run:
        _say_error("您的方案不覆蓋任何選定的探針",
                   f"您的方案包含：{', '.join(lic.packs)}。請在第 6 步取消勾選不在方案內的項目。")
        return

    # ── 4. 開始掃描 ──────────────────────────────────────────────────
    print()
    print("━" * 60)
    print(f"  開始執行 {len(will_run)} 個探針")
    print(f"  預估耗時：{len(will_run) * 5}–{len(will_run) * 10} 分鐘")
    print(f"  請保持本視窗開啟，過程中可隨時關閉再回來看結果")
    print("━" * 60)
    print()

    try:
        client = EndpointClient(
            endpoint_url=ENDPOINT_URL,
            api_key=API_KEY,
            provider=PROVIDER,
            model=MODEL,
            timeout=20,
        )
        t0 = time.time()
        report = run_scan(client=client, customer=lic.customer, packs=will_run, quiet=False)
        duration = time.time() - t0
    except Exception as e:
        msg = str(e).lower()
        if "401" in msg or "unauthorized" in msg or "invalid_api_key" in msg:
            _say_error("您的 API Key 無效或已撤銷",
                       "請至您的 AI 服務商後台（如 platform.openai.com/api-keys）重新生成 Key 後再試")
        elif "quota" in msg or "insufficient" in msg or "billing" in msg:
            _say_error("您的 AI 帳戶餘額不足",
                       "請至 AI 服務商後台儲值（OpenAI 充值 US$5 約可跑完整 9 探針）")
        elif "429" in msg or "rate" in msg:
            _say_error("AI 服務商限流（短時間請求太多）",
                       "請等 1–2 分鐘後重新點 ▶，或減少同時勾選的探針數量")
        elif "timeout" in msg or "connect" in msg or "dns" in msg:
            _say_error("無法連接到您的 AI 端點",
                       "請檢查端點 URL 是否正確、網路是否通暢、防火牆是否阻擋")
        else:
            _say_error("掃描過程出現問題", "請寄信描述您看到的情況")
        return

    # ── 5. 儲存報告 ──────────────────────────────────────────────────
    try:
        json_path = save_json(report, ".")
        pdf_path = save_pdf(report, ".")
        with open(json_path, encoding="utf-8") as f:
            data = json.load(f)
    except Exception:
        _say_error("報告生成失敗", "掃描資料已產生，請寄信協助取出")
        return

    # ── 6. 顯示分數總表 ──────────────────────────────────────────────
    print()
    print("━" * 60)
    print(f"  ✅ 掃描完成 — {data['customer']}")
    print("━" * 60)
    print(f"  總分：{data['overall_score']:.1f} / 100  （{data['overall_grade']} 級）")
    print(f"  耗時：{duration:.0f} 秒")
    print()
    for r in data["probe_results"]:
        bar_len = int(r["score"] / 5)
        bar = "█" * bar_len + "░" * (20 - bar_len)
        print(f"  {r['probe_name']:24s} {bar} {r['score']:5.1f}  ({r['grade']})")
    print()

    # ── 7. 自動下載 PDF + JSON 到客戶電腦 ────────────────────────────
    print("📥 正在下載 PDF 報告到您的電腦…")
    try:
        from google.colab import files
        files.download(pdf_path)
        files.download(json_path)
        print("✅ 已下載到您瀏覽器的預設下載資料夾")
    except Exception:
        print(f"   檔案位置：{pdf_path}（請手動下載）")
    print()
    print(f"如需協助解讀報告 / 安排修補諮詢，歡迎寄信至 {SUPPORT_EMAIL}")

_main()
